#Data Collection: Nigerian App Reviews
This notebook collects public Google Play reviews of Nigerian shopping apps (Jumia, Jiji).
It was run once on 20 November 2026 and the result was saved as `nigerian_app_reviews.csv`.
We wont re-run it, because Google Play reviews change every day.


## Collecting More Nigerian Reviews
We collect public Google Play reviews of Nigerian shopping apps to get more real complaint examples.


In [ ]:
# Install the library that collects public reviews from Google Play
!pip install google-play-scraper -q


In [ ]:
from google_play_scraper import search, reviews, Sort

# Let's find the official Google Play ID of each Nigerian shopping app
for app_name in ["Jumia", "Konga", "Jiji Nigeria"]:
    result = search(app_name, lang="en", country="ng", n_hits=1)
    print(app_name, "->", result[0]["appId"], "|", result[0]["title"])


Jumia -> None | JUMIA Online Shopping
Konga -> None | Konga Online Marketplace
Jiji Nigeria -> None | Jiji Nigeria: Buy&Sell Online


In [ ]:
# The top search result sometimes shows "None" for its ID (a known bug in the library),
# so let's list the top 5 results for each app and pick the official one
for app_name in ["Jumia", "Konga", "Jiji Nigeria"]:
    print("~" * 40)
    print(f" Results for {app_name}:")
    print("~" * 40)
    for result in search(app_name, lang="en", country="ng", n_hits=5):
        print(result["appId"], "|", result["title"])
    print()


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
 Results for Jumia:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
None | JUMIA Online Shopping
pt.jumia.services.delivery | Jumia Delivery Services
com.jumia.pus | Jumia Station App
com.einnovation.temu | Temu: Shop Like a Billionaire
com.amazon.mShop.android.shopping | Amazon Shopping

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
 Results for Konga:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
None | Konga Online Marketplace
com.kongapay.android | KongaPay
com.zzkko | SHEIN-Shopping Online
com.alibaba.aliexpresshd | AliExpress - Shopping App
com.jumia.android | JUMIA Online Shopping

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
 Results for Jiji Nigeria:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
None | Jiji Nigeria: Buy&Sell Online
com.justrite | Justrite App
com.jumia.android | JUMIA Online Shopping
com.nipek.cdcare | CDcare - Pay Small Small App
com.ebay.mobile | eBay online shopping & selling



In [ ]:
from google_play_scraper import app, reviews, Sort

# The Google Play IDs of the Nigerian shopping apps
# (Jumia's ID was confirmed in our search; Konga and Jiji are checked below before collecting)
shopping_apps = {
    "jumia_app": "com.jumia.android",
    "konga_app": "com.konga.android",
    "jiji_app": "ng.jiji.app",
}

all_reviews = []

for source_name, app_id in shopping_apps.items():
    # Let's first check that the app ID exists, and skip it if it doesn't
    try:
        details = app(app_id, lang="en", country="ng")
        print(f"Found: {details['title']} ({app_id})")
    except Exception:
        print(f"Skipped: {app_id} was not found")
        continue

    # Let's collect up to 400 reviews for each star rating (1 to 5),
    # so we get plenty of complaints (1-2 stars) and not just positive reviews
    for star in [1, 2, 3, 4, 5]:
        result, _ = reviews(app_id, lang="en", country="ng",
                            sort=Sort.NEWEST, count=400, filter_score_with=star)
        for r in result:
            # We keep only the review text and star rating (no usernames, to protect privacy)
            all_reviews.append({"text": r["content"], "rating": r["score"], "source": source_name})

# Let's put all the collected reviews into a table
app_reviews = pd.DataFrame(all_reviews)
print("\nTotal reviews collected:", app_reviews.shape)
app_reviews["source"].value_counts()


Found: JUMIA Online Shopping (com.jumia.android)
Skipped: com.konga.android was not found
Found: Jiji Nigeria: Buy&Sell Online (ng.jiji.app)

Total reviews collected: (4000, 3)


,count
source,
jumia_app,2000
jiji_app,2000


In [ ]:
# Let's label the reviews from their star ratings, the same rule we used for the Jumia data
# 1-2 stars = negative, 3 stars = neutral, 4-5 stars = positive
app_reviews["sentiment"] = app_reviews["rating"].map({1: "negative", 2: "negative", 3: "neutral",
                                                       4: "positive", 5: "positive"})

# Let's check the sentiment distribution and view some complaints
print(app_reviews["sentiment"].value_counts())
app_reviews[app_reviews["sentiment"] == "negative"].head(10)


sentiment
negative    1600
positive    1600
neutral      800
Name: count, dtype: int64


,text,rating,source,sentiment
0,app isn't working and it's not installing,1,jumia_app,negative
1,it's difficult to log in ur account,1,jumia_app,negative
2,"Hello Jumia, you put my order as ""Return in Pr...",1,jumia_app,negative
3,poor customer care poor refund people can't re...,1,jumia_app,negative
4,I requsted for return but I havn't recieve my ...,1,jumia_app,negative
5,Not bad,1,jumia_app,negative
6,once logged out u struggle to log in. like the...,1,jumia_app,negative
7,Why would you just cancel my order because of ...,1,jumia_app,negative
8,لديهم سياسة تسمح ببيع منتجات مغشوشة على انها ا...,1,jumia_app,negative
9,منتجات غير الصور اصغر و رديئة و مش نفس عدد القطع,1,jumia_app,negative


In [ ]:
# Some reviews are in Arabic (Jumia also operates in Egypt and Morocco), so we keep only
# reviews written mostly in English letters (English or Pidgin)
def is_english_or_pidgin(text):
    letters = [c for c in str(text) if c.isalpha()]
    if len(letters) == 0:
        return False
    english_letters = [c for c in letters if c.lower() in "abcdefghijklmnopqrstuvwxyz"]
    return len(english_letters) / len(letters) > 0.9     # at least 90% English letters

app_reviews = app_reviews[app_reviews["text"].apply(is_english_or_pidgin)]

# We also remove very short reviews (fewer than 3 words), e.g. "Not bad",
# because they carry too little information and are often mislabelled
app_reviews = app_reviews[app_reviews["text"].str.split().str.len() >= 3]

print("Reviews left:", app_reviews.shape)
print(app_reviews["sentiment"].value_counts())


Reviews left: (2633, 4)
sentiment
negative    1214
positive     929
neutral      490
Name: count, dtype: int64


In [ ]:
# Let's save the collected reviews so we don't need to collect them again
# (Google Play reviews change every day, so saving keeps our results the same)
app_reviews[["text", "rating", "sentiment", "source"]].to_csv("nigerian_app_reviews.csv", index=False)
print("Saved!")


Saved!
